# WFP Market Price Preprocessing

In [ ]:
import pandas as pd
from paths import PREPROCESSED_PATHS, PROJECT_START_DATE, RAW_PATHS, WFP_COMMODITIES, save_dataset


In [ ]:
df_wfp = pd.read_csv(RAW_PATHS['wfp_prices'])
df_wfp['date'] = pd.to_datetime(df_wfp['date'])
filtered = df_wfp[
    df_wfp['commodity'].isin(WFP_COMMODITIES)
    & (df_wfp['currency'] == 'LKR')
    & (df_wfp['date'] >= PROJECT_START_DATE)
].copy()
filtered['price'] = pd.to_numeric(filtered['price'], errors='coerce')
pivot = filtered.pivot_table(index='date', columns=['market', 'commodity'], values='price', aggfunc='mean')
pivot.columns = [f'{market}_{commodity}' for market, commodity in pivot.columns]
daily_idx = pd.date_range(start=PROJECT_START_DATE, end=pivot.index.max(), freq='D')
daily = pivot.reindex(daily_idx).ffill().bfill()
daily.index.name = 'Report_Date'
save_dataset(daily, PREPROCESSED_PATHS['wfp_daily'])
print('WFP daily shape:', daily.shape)
